In [1]:
import pandas as pd

train = pd.read_csv("train.csv")
train.columns = ["id", "class", "reactants>reagents>production"]
test = pd.read_csv("test.csv")
test.columns = ["id", "class", "reactants>reagents>production"]
valid = pd.read_csv("val.csv")
valid.columns = ["id", "class", "reactants>reagents>production"]
train.to_csv("raw_train.csv", index=False)
test.to_csv("raw_test.csv", index=False)
valid.to_csv("raw_val.csv", index=False)

# Remove reactions with only molecules with no atoms



In [27]:
import pandas as pd

train = pd.read_csv("train.csv")
train

,id,class,rxn_smiles
0,26182,UNK,CON(C)[C:2](=[O:3])[CH:4]1[CH2:5][CH2:6][N:7](...
1,86857,UNK,CC(C)(C)OC(=O)[NH:21][C@H:20]1[CH2:19][CH2:18]...
2,76265,UNK,Cl[S:5](=[O:6])(=[O:7])[CH:8]=[CH:9][c:10]1[cH...
3,52534,UNK,CC(C)=[C:2]1[CH:3]2[CH2:4][CH2:5][CH:6]1[c:7]1...
4,58325,UNK,[CH3:1][C@H:5]([NH2:6])[C:7](=[O:8])[OH:9]>>[C...
...,...,...,...
73188,6441,UNK,[CH3:1][O:2][CH2:3][O:4][c:5]1[c:6]([Cl:7])[cH...
73189,56565,UNK,S=[C:21]1[C:19]([CH3:20])=[N:18][C:17]2([C:6]3...
73190,79172,UNK,C[O:12][C:10]([c:9]1[c:7]([F:8])[cH:6][cH:5][c...
73191,885,UNK,Cl[S:11]([c:10]1[c:2]([CH3:1])[cH:3][cH:4][c:5...


In [43]:
from rdkit import Chem

def check_smiles(smiles):

    # for smiles_ in smiles.split("."):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return False
    
    Chem.MolToSmiles(mol, isomericSmiles=True, rootedAtAtom=34, canonical=True)
    return True

In [45]:
check_smiles("[O-:1][P:2](=[O:3])([OH:4])[O-:5].[CH2:6]([C@@H:7]1[C@@H:8]([OH:21])[C@@H:9]([OH:20])[C@H:10]([N:11]2[C:12](=[O:18])[N:13]=[C:14]([NH2:17])[CH:15]=[CH:16]2)[O:19]1)[OH:22].[CH2:23]([C@@H:24]1[C@@H:25]([OH:38])[C@@H:26]([OH:37])[C@H:27]([N:28]2[C:29](=[O:35])[N:30]=[C:31]"
"([NH2:34])[CH:32]=[CH:33]2)[O:36]1)[OH:39].[O-:40][P:41](=[O:42])([OH:43])[O-:44]")

[16:26:28] 

****
Range Error
idx
Violation occurred on line 209 in file /project/build/temp.linux-x86_64-cpython-310/rdkit/Code/GraphMol/ROMol.cpp
Failed Expression: 34 < 5
----------
Stacktrace:
----------
****



RuntimeError: Range Error
	idx
	Violation occurred on line 209 in file Code/GraphMol/ROMol.cpp
	Failed Expression: 34 < 5
	RDKIT: 2024.03.6
	BOOST: 1_85


In [29]:
from rdkit import Chem
from tqdm import tqdm

def remove_empty_invalid_molecules(dataset):
    to_drop = []

    for i, row in tqdm(dataset.iterrows(), total=len(dataset), desc="Removing empty or invalid molecules"):
        smiles = row["rxn_smiles"]
        reactants, products = smiles.split(">>")
        valid = check_smiles(reactants)
        if not valid:
            to_drop.append(i)
        valid = check_smiles(products)
        if not valid:
            to_drop.append(i)
    to_drop = set(to_drop)  # Remove duplicates
    return dataset.drop(to_drop)
            


In [ ]:
train_cleaned = remove_empty_invalid_molecules(train)
train_cleaned

In [31]:
print(f"Removed {len(train) - len(train_cleaned)} empty or invalid reactions from training set.")

Removed 0 empty or invalid reactions from training set.
